# 02b · Preparação dos dados e sensibilidade à escala

**CRISP-DM fase 3**

O experimento que responde à pergunta central da preparação para algoritmos baseados
em distância: **quanto do resultado é dado, e quanto é escolha de pré-processamento?**

Quatro versões do mesmo espaço de atributos, o mesmo K-Means, o mesmo *k*. Se as
partições resultantes discordarem, quem definiu os segmentos foi o escalonador.

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.data import dicionario as dic
from src.data.preparacao import (
    REGRAS_LIMPEZA,
    carregar_curada,
    construir_matriz_modelagem,
    preparar_espaco,
)

curada = carregar_curada()
matriz = construir_matriz_modelagem(curada, salvar=False)
pd.DataFrame(REGRAS_LIMPEZA)

## O espaço de atributos

21 variáveis: 11 numéricas, 6 ordinais e 4 binárias derivadas. As nominais ficam de
fora do espaço euclidiano — a comparação com Gower está no notebook 03.

In [ ]:
espaco, nomes = preparar_espaco(matriz, "padrao")
print(f"{espaco.shape[0]} imóveis × {espaco.shape[1]} variáveis")
pd.DataFrame(
    {
        "variável": [dic.rotulo(n) for n in nomes],
        "coluna": nomes,
        "log1p": ["sim" if n in dic.ASSIMETRICAS else "—" for n in nomes],
        "descrição": [dic.DESCRICOES.get(n, "—") for n in nomes],
    }
)

## Por que `log1p`, e só em três variáveis

O limiar é 0,75 de assimetria absoluta — convenção usual. Abaixo dele, o ganho não
compensa a perda de legibilidade da unidade original. A área do porão fica de fora
(0,40): transformá-la seria hábito, não decisão.

In [ ]:
assimetria = curada[dic.NUMERICAS].skew().abs().sort_values(ascending=False)
pd.DataFrame(
    {
        "assimetria": assimetria.round(2),
        "acima do limiar": np.where(assimetria > dic.LIMIAR_ASSIMETRIA, "sim → log1p", "—"),
    }
)

## O experimento de sensibilidade

Quatro escalonadores, o mesmo K-Means com *k* = 4, e o índice de Rand ajustado entre
cada par de partições resultantes.

In [ ]:
from src.model.estabilidade import sensibilidade_escala

resumo, ari = sensibilidade_escala(matriz, k=4)
resumo

In [ ]:
ari

## A armadilha

Repare que **`sem_escala` tem a maior silhueta**. Não é um bom resultado: é o sintoma.

Sem padronização, a variável com maior amplitude numérica domina a distância
euclidiana, e o K-Means acaba particionando por ela sozinha. A silhueta fica alta
porque um agrupamento praticamente unidimensional é, de fato, bem separado — só que
não é a segmentação que a diretoria pediu.

A tabela abaixo mostra, para cada escalonador, **por qual atributo o algoritmo de fato
particionou**, medido pela razão entre a variância entre grupos e a variância dentro
dos grupos.

In [ ]:
from src.pipeline.modelagem import _atributo_dominante
from src.data import preparacao
from sklearn.cluster import KMeans

linhas = []
for escalador in preparacao.ESCALADORES:
    dados, _ = preparar_espaco(matriz, escalador=escalador)
    rotulos = KMeans(n_clusters=4, n_init=20, random_state=config.SEMENTE).fit_predict(dados)
    atributo, razao = _atributo_dominante(curada, rotulos)
    linhas.append(
        {
            "escalador": escalador,
            "silhueta": round(resumo.loc[escalador, "silhueta"], 4),
            "atributo dominante": atributo,
            "razão entre/dentro": round(razao, 2),
        }
    )
pd.DataFrame(linhas).set_index("escalador")

## Conclusão do experimento

O `StandardScaler` é o escolhido. Não por ter a melhor métrica — não tem —, mas
porque é o único em que **nenhuma variável isolada domina a geometria**, e o resultado
é uma segmentação multivariada de fato.

É a demonstração prática de que **silhueta mais alta não significa segmentação melhor**.
A métrica mede separação geométrica; ela não sabe se o que separou faz sentido de
negócio.